# 📉 Day 1 — Project Setup & Data Exploration

**Project**: Customer Churn Prediction  
**Dataset**: IBM Telco Customer Churn (Kaggle)  
**Date**: 2026-08-06  

---

## 🎯 Objectives for Today

1. Load and inspect the raw dataset
2. Understand each column's meaning and data type
3. Identify data quality issues (missing values, wrong types)
4. Understand the target variable distribution
5. Capture initial statistical summaries

> **Note**: We are only *exploring* today — no transformations yet. That's Day 2.

## 1. 📦 Import Libraries

In [ ]:
import sys
import os

# Add project root to path so we can import from src/
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    RAW_DATA_PATH,
    TARGET_COL,
    NUMERICAL_COLS,
    PLOT_STYLE,
    FIG_DPI,
    COLOR_CHURN,
    IMAGES_DIR
)

# Display settings
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.style.use(PLOT_STYLE)

print('✅ Libraries imported successfully')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')
print(f'   seaborn : {sns.__version__}')

## 2. 📂 Load the Dataset

In [ ]:
# Load raw data using the path from config
df = pd.read_csv(RAW_DATA_PATH)

print(f'✅ Dataset loaded successfully!')
print(f'   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

## 3. 🔍 Dataset Structure & Data Types

In [ ]:
print('=' * 60)
print('DATASET INFO')
print('=' * 60)
df.info()

### 📋 Column Dictionary

| Column | Type | Description |
|--------|------|-------------|
| `customerID` | str | Unique customer identifier (drop before modeling) |
| `gender` | str | Male / Female |
| `SeniorCitizen` | int | 1 = Senior, 0 = Non-Senior |
| `Partner` | str | Has a partner: Yes / No |
| `Dependents` | str | Has dependents: Yes / No |
| `tenure` | int | Months with the company |
| `PhoneService` | str | Has phone service: Yes / No |
| `MultipleLines` | str | Multiple lines: Yes / No / No phone service |
| `InternetService` | str | DSL / Fiber optic / No |
| `OnlineSecurity` | str | Yes / No / No internet service |
| `OnlineBackup` | str | Yes / No / No internet service |
| `DeviceProtection` | str | Yes / No / No internet service |
| `TechSupport` | str | Yes / No / No internet service |
| `StreamingTV` | str | Yes / No / No internet service |
| `StreamingMovies` | str | Yes / No / No internet service |
| `Contract` | str | Month-to-month / One year / Two year |
| `PaperlessBilling` | str | Yes / No |
| `PaymentMethod` | str | 4 payment options |
| `MonthlyCharges` | float | Monthly bill amount |
| `TotalCharges` | **str** ⚠️ | Should be float — has blank values! |
| `Churn` | str | **TARGET**: Yes = churned, No = stayed |

## 4. ❓ Missing Value Analysis

The standard `isnull()` check will show zeros — but `TotalCharges` has a known issue where blank strings (`' '`) are stored instead of NaN. Let's catch both.

In [ ]:
print('--- Standard NaN Check ---')
missing_standard = df.isnull().sum()
print(missing_standard[missing_standard > 0] if missing_standard.sum() > 0 else 'No NaN values found')
print()

# Check for hidden blank strings in object columns
print('--- Hidden Blank String Check ---')
object_cols = df.select_dtypes(include='object').columns
for col in object_cols:
    blank_count = (df[col].str.strip() == '').sum()
    if blank_count > 0:
        print(f'  ⚠️  "{col}" has {blank_count} blank/whitespace values')

print()
# Specifically inspect TotalCharges
print('--- TotalCharges sample of problematic rows ---')
problematic = df[df['TotalCharges'].str.strip() == '']
print(f'Count of blank TotalCharges: {len(problematic)}')
print(problematic[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']])

### 💡 Finding
- `TotalCharges` appears as `object` (string) instead of `float64`
- **11 rows** have blank `TotalCharges` values — these correspond to customers with `tenure = 0` (brand new customers who haven't been billed yet)
- **Fix (Day 2)**: Convert `TotalCharges` → float, replace blanks with `NaN`, then impute with median

## 5. 📊 Duplicate Check

In [ ]:
dup_count = df.duplicated().sum()
print(f'Duplicate rows: {dup_count}')

# Check if customerID is truly unique
print(f'Total rows           : {len(df):,}')
print(f'Unique customerIDs   : {df["customerID"].nunique():,}')
print(f'→ customerID is unique: {len(df) == df["customerID"].nunique()}')

## 6. 🎯 Target Variable Analysis — Churn Distribution

In [ ]:
churn_counts = df[TARGET_COL].value_counts()
churn_pct    = df[TARGET_COL].value_counts(normalize=True) * 100

print('Churn Distribution:')
print('-' * 30)
for label in churn_counts.index:
    print(f'  {label:5s}: {churn_counts[label]:5,}  ({churn_pct[label]:.1f}%)')
print()

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Customer Churn Distribution', fontsize=16, fontweight='bold', y=1.02)

colors = [COLOR_CHURN['No'], COLOR_CHURN['Yes']]

# Bar chart
ax1 = axes[0]
bars = ax1.bar(churn_counts.index, churn_counts.values, color=colors, edgecolor='white', linewidth=1.5, width=0.5)
ax1.set_title('Count of Churned vs Retained', fontsize=13)
ax1.set_xlabel('Churn', fontsize=11)
ax1.set_ylabel('Number of Customers', fontsize=11)
for bar, (label, count, pct) in zip(bars, zip(churn_counts.index, churn_counts.values, churn_pct.values)):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
             f'{count:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax1.set_ylim(0, churn_counts.max() * 1.2)

# Pie chart
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    churn_counts.values,
    labels=churn_counts.index,
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    explode=(0, 0.05),
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
for text in autotexts:
    text.set_fontsize(12)
    text.set_fontweight('bold')
ax2.set_title('Churn Proportion', fontsize=13)

plt.tight_layout()
plt.savefig(IMAGES_DIR / 'day1_churn_distribution.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()
print(f'\n✅ Plot saved to images/day1_churn_distribution.png')

### 💡 Finding
- The dataset is **imbalanced**: ~73.5% retained vs ~26.5% churned
- This means **accuracy is misleading** — a dummy model predicting 'No' gets 73.5%!
- We'll need to use **ROC-AUC, F1, and Recall** as primary metrics
- Will use **SMOTE** or `class_weight='balanced'` to handle imbalance

## 7. 🔢 Numerical Features — Statistical Summary

In [ ]:
# Note: TotalCharges is object, convert temporarily for stats
df_num = df[NUMERICAL_COLS].copy()
df_num['TotalCharges'] = pd.to_numeric(df_num['TotalCharges'], errors='coerce')

print('Numerical Features — Descriptive Statistics')
print('=' * 60)
display(df_num.describe().T.round(2))

In [ ]:
# Distribution plots for numerical features
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Numerical Feature Distributions', fontsize=16, fontweight='bold')

num_cols_plot = NUMERICAL_COLS  # ['tenure', 'MonthlyCharges', 'TotalCharges']

for i, col in enumerate(num_cols_plot):
    col_data = pd.to_numeric(df[col], errors='coerce').dropna()
    
    # Histogram + KDE
    ax_hist = axes[0][i]
    ax_hist.hist(col_data, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    ax_hist.set_title(f'{col} — Histogram', fontsize=12)
    ax_hist.set_xlabel(col)
    ax_hist.set_ylabel('Count')
    
    # Boxplot by Churn
    ax_box = axes[1][i]
    df_temp = df[['Churn', col]].copy()
    df_temp[col] = pd.to_numeric(df_temp[col], errors='coerce')
    
    groups = [df_temp[df_temp['Churn'] == label][col].dropna() for label in ['No', 'Yes']]
    bp = ax_box.boxplot(groups, labels=['No (Stayed)', 'Yes (Churned)'],
                        patch_artist=True, notch=False)
    for patch, color in zip(bp['boxes'], [COLOR_CHURN['No'], COLOR_CHURN['Yes']]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax_box.set_title(f'{col} — by Churn', fontsize=12)
    ax_box.set_ylabel(col)

plt.tight_layout()
plt.savefig(IMAGES_DIR / 'day1_numerical_distributions.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()
print('✅ Plot saved to images/day1_numerical_distributions.png')

### 💡 Findings — Numerical Features

| Feature | Key Observation |
|---------|----------------|
| `tenure` | Wide distribution (1–72 months). **Churned customers have much lower tenure** — they leave early |
| `MonthlyCharges` | **Churned customers pay more** monthly — likely on Fiber Optic plans |
| `TotalCharges` | Strongly correlated with tenure. Churners have lower totals because they leave early |

## 8. 🔤 Categorical Features — Value Counts

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c not in ['customerID', TARGET_COL]]

print(f'Categorical columns ({len(cat_cols)}):')
print('-' * 40)
for col in cat_cols:
    vals = df[col].value_counts()
    print(f'\n▶ {col} ({df[col].nunique()} unique):')
    for val, cnt in vals.items():
        pct = cnt / len(df) * 100
        print(f'    {val:<30} {cnt:>5,}  ({pct:.1f}%)')

In [ ]:
# Quick visual: churn rate per categorical feature
# Show top-level binary/small-cardinality columns
cols_to_plot = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'Contract', 'InternetService', 'PaymentMethod', 'PaperlessBilling'
]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Churn Rate by Key Categorical Features', fontsize=16, fontweight='bold')

for idx, col in enumerate(cols_to_plot):
    ax = axes[idx // 4][idx % 4]
    
    col_data = df[[col, TARGET_COL]].copy()
    col_data[TARGET_COL] = col_data[TARGET_COL].map({'No': 0, 'Yes': 1})
    
    # Churn rate per category
    churn_rate = col_data.groupby(col)[TARGET_COL].mean().sort_values(ascending=False) * 100
    
    bars = ax.bar(churn_rate.index.astype(str), churn_rate.values,
                  color='#E57373', edgecolor='white', linewidth=1.2, alpha=0.85)
    ax.set_title(f'{col}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Churn Rate (%)', fontsize=10)
    ax.set_xticklabels(churn_rate.index.astype(str), rotation=30, ha='right', fontsize=8)
    ax.set_ylim(0, min(100, churn_rate.max() * 1.4))
    ax.axhline(y=26.5, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='Overall avg')
    
    for bar, val in zip(bars, churn_rate.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig(IMAGES_DIR / 'day1_categorical_churn_rates.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()
print('✅ Plot saved to images/day1_categorical_churn_rates.png')

### 💡 Findings — Categorical Features

| Feature | Key Observation |
|---------|----------------|
| `Contract` | **Month-to-month** customers churn at ~43% vs Two year at ~3% — strongest signal! |
| `InternetService` | **Fiber optic** customers churn at ~42% — possibly due to higher prices |
| `PaymentMethod` | **Electronic check** users churn most (~45%) |
| `PaperlessBilling` | Paperless billing customers churn more (~33%) |
| `SeniorCitizen` | Seniors have ~41% churn rate vs ~24% for non-seniors |
| `Partner` / `Dependents` | Customers without dependents/partner churn more |

## 9. 📝 Day 1 Summary — Key Findings

### ✅ What We Learned

**Dataset:**
- 7,043 customers, 21 columns, 0 true NaN values
- No duplicate rows
- `customerID` is unique → drop before modeling

**Data Issues to Fix (Day 2):**
- `TotalCharges` → stored as string, 11 blank rows need imputation
- `SeniorCitizen` → already 0/1 int, no encoding needed

**Target Variable:**
- Imbalanced: 73.5% No (retained) vs 26.5% Yes (churned)
- Must use F1/ROC-AUC as primary metrics, not accuracy

**Top Churn Predictors (Preliminary):**
1. `Contract` — Month-to-month is high risk
2. `InternetService` — Fiber optic users churn most
3. `tenure` — Lower tenure → higher churn risk
4. `MonthlyCharges` — Higher bills correlate with churn
5. `PaymentMethod` — Electronic check users churn most

---

### 🔜 Tomorrow (Day 2): Data Cleaning
- Fix `TotalCharges` type conversion
- Impute blank `TotalCharges` values
- Remove `customerID`
- Save cleaned CSV to `data/processed/cleaned_data.csv`